# 1. Generate, validate and understand the data

This is a controlled optical-loss simulator, not measured operator data.
Start with `pip install -e ".[dev]"` from the repository root.
The single YAML describes simulation assumptions. Generated files stay outside Git.
The adapter keeps only identifiers, time and the two optical powers used by this model.
See the methodology for assumptions and references.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from telco_anomaly.synthetic import load_config
from telco_anomaly.experiment import split_times, load_data

CONFIG, relative_path = load_config(ROOT / "configs/synthetic.yml")
DATA = ROOT / relative_path
RUN = ROOT / "outputs/simple_model"


In [ ]:
from dataclasses import asdict
from telco_anomaly.synthetic import generate_dataset
from telco_anomaly.synthetic_validation import validate_dataset
from telco_anomaly.adapter import quality_report

if not DATA.exists():
    generate_dataset(CONFIG, DATA)
manifest = json.loads((DATA / "manifest.json").read_text())
assert manifest["config"] == asdict(CONFIG), "Use a new data path after changing settings"
report = validate_dataset(DATA)
display(report["checks"])
assert report["checks"].status.eq("pass").all()


## EDA uses the reference period only

Validation checks mathematical and structural consistency; it cannot prove realism.
Fault labels are evaluation sidecars and are never adapter or model inputs.
Inspect missingness before interpreting apparent recovery. Do not forward-fill gaps.

In [ ]:
_, times = split_times(DATA)
training = load_data(DATA, times["fit_end"])
display(training.describe(include="number"))
display(quality_report(training, CONFIG.sample_minutes).head(12))
entity = training.ont_id.iloc[0]
one = training.loc[training.ont_id.eq(entity)].set_index("timestamp_utc")
one[["rx_power_dbm", "olt_rx_power_dbm"]].plot(
    figsize=(12, 4), title=f"Reference optical powers: {entity}", ylabel="dBm"
)
plt.show()


## Seasonality is inspected before adding a seasonal model

Centre each device to avoid mixing different link budgets. Daily patterns may be
normal temperature effects. These UTC profiles are descriptive, not proof that
removing seasonality improves detection. For a company dataset use its operational
timezone and multiple cycles; add a seasonal correction only if subsequent data
shows fewer false alarms without losing gradual-fault warnings. The current detector
uses a robust spread and smoothing; it does not explicitly remove seasonality.

In [ ]:
profiles = training.copy()
for metric in ["rx_power_dbm", "olt_rx_power_dbm"]:
    profiles[metric] -= profiles.groupby("ont_id")[metric].transform("median")
profiles["hour"] = profiles.timestamp_utc.dt.hour
profiles.groupby("hour")[["rx_power_dbm", "olt_rx_power_dbm"]].median().plot(
    figsize=(9, 3), ylabel="Deviation from device median (dB)",
    title="Typical daily profile in reference data"
)
plt.show()
